In [1]:
# Get raw data
with open('input/23.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
import numpy as np

In [3]:
# Part 1
def yxlist(trailmap):
    return np.column_stack((np.repeat(np.arange(trailmap.shape[0]), 
                                      trailmap.shape[1]),
                            np.tile(np.arange(trailmap.shape[1]), 
                                    trailmap.shape[0])))

dir_lkp = [('v',2,None,1,-1),
           ('^',None,-2,1,-1),
           ('>',1,-1,2,None),
           ('<',1,-1,None,-2)]
pathexp = lambda x,tm,kpts: np.max(np.stack([np.where((x<0)
                                                      & (tm != '#')
                                                      & (tm != i[0])
                                                      & ((z:=rp[i[1]:i[2],i[3]:i[4]])>=0)
                                                      & (((np.pad(kpts,1)[i[1]:i[2],i[3]:i[4]]-2)*z)==0),
                                                      z+1,
                                                      x)
                                             for rp in [np.pad(x,1,constant_values=-1)]
                                             for i in dir_lkp]),
                                   axis=0)

def get_paths(startpt,trailmap,kpts):
    reach = -np.ones(trailmap.shape, dtype=int)
    reach[*startpt] = 0
    while np.any(reach != (reach:=pathexp(reach,trailmap,kpts))):  pass
    return [(tuple(j),i) 
            for z in [np.where((reach>=0)&(kpts>2),reach,-1).reshape(-1)]
            for i,j in zip(z[z>0].tolist(), yxlist(trailmap)[z>0].tolist())]
    
def get_edges(trailmap):
    kpts = ((z:=np.pad((trailmap != '#').astype(int), 1, constant_values=2))[1:-1,1:-1] 
            * np.sum(np.stack((z[:-2,1:-1], z[2:,1:-1], z[1:-1,:-2], z[1:-1,2:])), 
                     axis=0))
    kpix = {tuple(j):i for i,j in enumerate(yxlist(trailmap)[kpts.reshape(-1)>2].tolist())}
    
    rawedges = [[(kpix[j],k) 
                 for j,k in get_paths(i,trailmap,kpts)]
                for i in sorted(kpix)]
    w = max([len(i) for i in rawedges])
    
    return [np.array([([j[h] for j in i]+[-1]*w)[:w] for i in rawedges])
            for h in [0,1]]
    
def get_longest_path(trailmap):
    edge_end, edge_len = get_edges(trailmap)
    target = edge_end.shape[0]-1
    w = edge_end.shape[1]
    
    pos = np.array([0])
    hist = np.empty((1,0), dtype=int)
    nsteps = np.array([0])
    max_n_steps = 0
    while pos.shape[0]:
        pos, hist, nsteps = (edge_end[pos].reshape(-1),
                             np.repeat(np.sort(np.column_stack((hist,pos)), axis=1), w, axis=0),
                             np.repeat(nsteps, w) + edge_len[pos].reshape(-1))
        filt = (pos != -1) & np.all(np.expand_dims(pos,1) != hist, axis=1)
        pos, hist, nsteps = [i[filt] for i in [pos, hist, nsteps]]
        
        sortix = np.lexsort([-nsteps, 
                             *[hist[:,i] for i in np.arange(hist.shape[1]-1,-1,-1)], 
                             pos])
        filt = np.any((z:=np.column_stack((pos[sortix], hist[sortix]))) 
                      != np.append(-np.ones_like(z[:1]), z[:-1], axis=0), 
                      axis=1)[np.argsort(sortix)]
        pos, hist, nsteps = [i[filt] for i in [pos, hist, nsteps]]
        
        if np.any(pos==target) and (z:=np.max(nsteps[pos==target]).item()) > max_n_steps:
            max_n_steps = z
            
    return max_n_steps
        
trailmap = np.array([[*i] for i in rawinput.split('\n')])
get_longest_path(trailmap)

1930

In [4]:
# Part 2
get_longest_path(np.where(trailmap != '#','.','#'))

6230